<a href="https://colab.research.google.com/github/Dina-Shanjida/Plant_Village/blob/main/Plant_Village.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms
from torch.utils.data import Dataset,DataLoader, Subset
from PIL import Image
import kagglehub

In [2]:
path = kagglehub.dataset_download("mohitsingh1804/plantvillage")
print(path)

Using Colab cache for faster access to the 'plantvillage' dataset.
/kaggle/input/plantvillage


In [3]:
torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cuda


In [4]:
TRAIN_PATH = os.path.join(path, "PlantVillage", "train")
VAL_PATH = os.path.join(path, "PlantVillage", "val")


In [5]:
transform = transforms.Compose(
    [
        transforms.Resize((128 , 128)),
        transforms.ToTensor(),
        transforms.Normalize(mean= [0.5, 0.5 , 0.5], std= [0.5,0.5,0.5])

    ]
)

In [7]:
from genericpath import isfile

class MultiClassClassification(Dataset):
  def __init__(self, root_dir , transform= None):
    super().__init__()

    self.samples = []
    self.transform = transform

    self.classes = sorted([d for d in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir,d))])
    self.class_to_idx = {cls_name: idx for idx, cls_name in enumerate(self.classes)}

    for class_name in self.classes:
      class_path = os.path.join(root_dir , class_name)

      for img_name in os.listdir(class_path):
        img_path = os.path.join(class_path, img_name)

        if os.path.isfile(img_path):
          label = self.class_to_idx[class_name]
          self.samples.append((img_path, label))

  def __len__(self):
    return len(self.samples)


  def __getitem__(self, idx):
    img_path , label = self.samples[idx]
    image = Image.open(img_path).convert("RGB")
    if self.transform:
      image = self.transform(image)

    return image , label

In [8]:
train_dataset = MultiClassClassification(TRAIN_PATH , transform)
test_dataset = MultiClassClassification(VAL_PATH, transform)
num_classes = len(train_dataset.classes)

print("Total classes:", num_classes)
print("Train size:", len(train_dataset))
print("Test size:", len(test_dataset))


Total classes: 38
Train size: 43444
Test size: 10861


In [9]:
pin = True if device.type == 'cuda' else False
train_loader = DataLoader(train_dataset , batch_size = 32 , shuffle = True)
test_loader = DataLoader(test_dataset , batch_size = 32 , shuffle = False)


In [16]:
class CustomCNN(nn.Module):
  def __init__(self , num_classes):
    super().__init__()
    self.features = nn.Sequential(
        nn.Conv2d(3, 32 , kernel_size = 3 , padding = 'same'),
        nn.ReLU(),
        nn.BatchNorm2d(32),
        nn.MaxPool2d(2),

        nn.Conv2d(32 , 64 , kernel_size = 3, padding = 'same'),
        nn.ReLU(),
        nn.BatchNorm2d(64),
        nn.MaxPool2d(2),

        nn.Conv2d(64 , 128 , kernel_size = 3 , padding = 'same'),
        nn.ReLU(),
        nn.BatchNorm2d(128),
        nn.MaxPool2d(2),

        nn.Conv2d(128 , 256 , kernel_size = 3 , padding = 'same'),
        nn.ReLU(),
        nn.BatchNorm2d(256),
        nn.MaxPool2d(2)
    )

    self.classifier = nn.Sequential(
        nn.Flatten(),
        nn.Linear(256*8*8 , 256),
        nn.ReLU(),
        nn.Dropout(0.5),

        nn.Linear(256, 64),
        nn.ReLU(),
        nn.Dropout(0.5),

        nn.Linear(64, num_classes)
    )

  def forward(self , x):
    x = self.features(x)
    x = self.classifier(x)
    return x

In [17]:
model = CustomCNN(num_classes = num_classes).to(device)
learning_rate = 0.001
epochs = 5
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr = learning_rate)

In [18]:
def train_one_epoch(model , train_loader , criterion , optimizer , device):

  model.train()

  total_loss = 0
  correct = 0
  total = 0

  for batch_features , batch_labels in train_loader:
    batch_features = batch_features.to(device)
    batch_labels = batch_labels.to(device)

    outputs = model(batch_features)

    loss = criterion(outputs , batch_labels)
    optimizer.zero_grad()

    loss.backward()

    optimizer.step()

    total_loss += loss.item()

    _, predicted = torch.max(outputs , 1)

    total += batch_labels.size(0)
    correct += (predicted == batch_labels).sum().item()

  avg_loss = total_loss / len(train_loader)

  accuracy = 100 * correct / total

  return avg_loss , accuracy





In [19]:
def validate_one_epoch(model, val_loader, criterion, device):

    model.eval()

    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():

        for batch_features, batch_labels in val_loader:

            batch_features = batch_features.to(device)
            batch_labels = batch_labels.to(device)

            outputs = model(batch_features)

            loss = criterion(outputs, batch_labels)

            total_loss += loss.item()


            _, predicted = torch.max(outputs, 1)

            total += batch_labels.size(0)

            correct += (predicted == batch_labels).sum().item()

    avg_loss = total_loss / len(val_loader)

    accuracy = 100 * correct / total

    return avg_loss, accuracy

In [21]:
for epoch in range(epochs):

    train_loss, train_acc = train_one_epoch(
        model,
        train_loader,
        criterion,
        optimizer,
        device
    )

    val_loss, val_acc = validate_one_epoch(
        model,
        test_loader,
        criterion,
        device
    )

    print(
        f"Epoch {epoch+1}/{epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.2f}% | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.2f}%"
    )

Epoch 1/5 | Train Loss: 2.0550 | Train Acc: 44.52% | Val Loss: 1.0876 | Val Acc: 68.85%
Epoch 2/5 | Train Loss: 1.2997 | Train Acc: 62.37% | Val Loss: 0.7981 | Val Acc: 74.79%
Epoch 3/5 | Train Loss: 1.0078 | Train Acc: 70.46% | Val Loss: 0.4952 | Val Acc: 84.86%
Epoch 4/5 | Train Loss: 0.8013 | Train Acc: 76.48% | Val Loss: 0.3738 | Val Acc: 89.26%
Epoch 5/5 | Train Loss: 0.6419 | Train Acc: 81.51% | Val Loss: 0.3381 | Val Acc: 89.53%
